In [1]:
import numpy as np
import math
import random

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# ============================================================
# 1. INPUT DATA (17 points)
# ============================================================

X = np.array([
    [0.66579958, 0.12396913],
    [0.87779099, 0.77862750],
    [0.14269907, 0.34900513],
    [0.84527543, 0.71112027],
    [0.45464714, 0.29045518],
    [0.57771284, 0.77197318],
    [0.43816606, 0.68501826],
    [0.34174959, 0.02869772],
    [0.33864816, 0.21386725],
    [0.70263656, 0.92656420],
    [0.92656400, 1.02656400],
    [0.74750400, 0.20897000],
    [0.68340600, 0.06376900],
    [0.58313700, 0.01254900],
    [0.99145700, 0.00174400],
    [0.99145700, 0.00174400],
    [0.99947700, 0.02152800]
], dtype=float)

y = np.array([
    0.53899612, 0.42058624, -0.06562362, 0.29399291, 0.21496451,
    0.02310555, 0.24461934, 0.03874902, -0.01385762, 0.61120522,
    -0.04199554, 0.28033031, 0.62973064, 0.06695726,
    0.11670354823827367, 0.11353912028668156, 0.02426575623315439
], dtype=float)

X = np.clip(X, 0, 1)

# ============================================================
# 2. DEDUPLICATION (average y for identical X)
# ============================================================

def dedupe_average(X, y):
    b = np.ascontiguousarray(X).view(
        np.dtype((np.void, X.dtype.itemsize * X.shape[1]))
    )
    _, inv = np.unique(b, return_inverse=True)

    Xu, yu = [], []
    for i in np.unique(inv):
        idx = np.where(inv == i)[0]
        Xu.append(X[idx[0]])
        yu.append(y[idx].mean())

    return np.array(Xu), np.array(yu)

X, y = dedupe_average(X, y)

# ============================================================
# 3. TRAIN-ONLY NOISE
# ============================================================

class TrainOnlyNoise:
    def __init__(self, sigma=0.005, seed=0):
        self.sigma = sigma
        self.seed = seed

    def fit(self, X, y=None):
        return self

    def fit_transform(self, X, y=None):
        rng = np.random.default_rng(self.seed)
        return X + rng.normal(0, self.sigma, size=X.shape)

    def transform(self, X):
        return X


def make_model(seed, hidden=(64, 32), alpha=5e-5, lr=0.01,
               sigma=0.005, max_iter=5000, tol=1e-7, n_iter_no_change=40):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("noise", TrainOnlyNoise(sigma=sigma, seed=seed)),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=hidden,
            activation="relu",
            solver="adam",
            learning_rate="adaptive",
            learning_rate_init=lr,
            alpha=alpha,
            early_stopping=True,
            n_iter_no_change=n_iter_no_change,
            max_iter=max_iter,
            tol=tol,
            random_state=seed
        ))
    ])

# ============================================================
# 4. CV RANDOM SEARCH
# ============================================================

def cv_mse_for_config(X, y, cfg):
    kf = KFold(n_splits=5, shuffle=True, random_state=123)
    mses = []

    for tr, te in kf.split(X):
        preds = []
        for i in range(cfg["ens_cv"]):
            m = make_model(
                seed=1000 + i,
                hidden=cfg["hidden"],
                alpha=cfg["alpha"],
                lr=cfg["lr"],
                sigma=cfg["sigma"],
                max_iter=cfg["max_iter"],
                tol=cfg["tol"],
                n_iter_no_change=cfg["ninc"]
            )
            m.fit(X[tr], y[tr])
            preds.append(m.predict(X[te]))

        mses.append(mean_squared_error(y[te], np.mean(preds, axis=0)))

    return float(np.mean(mses))


def random_search_best_config(X, y, n_trials=60, seed=8):
    random.seed(seed)

    best_cfg, best_mse = None, float("inf")

    for _ in range(n_trials):
        cfg = {
            "hidden": random.choice([(32,16),(64,32),(64,64),(128,64)]),
            "alpha": random.choice([1e-6,1e-5,5e-5,1e-4]),
            "lr": random.choice([3e-3,1e-2,2e-2]),
            "sigma": random.choice([0.0,0.003,0.005,0.01]),
            "tol": random.choice([1e-6,1e-7]),
            "ninc": random.choice([30,40,60]),
            "max_iter": random.choice([3000,5000]),
            "ens_cv": 5
        }

        mse = cv_mse_for_config(X, y, cfg)
        if mse < best_mse:
            best_cfg, best_mse = cfg, mse

    return best_cfg, best_mse


best_cfg, best_cv_mse = random_search_best_config(X, y)

# ============================================================
# 5. FINAL ENSEMBLE
# ============================================================

def fit_ensemble(X, y, cfg, n=24):
    models = []
    for i in range(n):
        m = make_model(
            seed=300+i,
            hidden=cfg["hidden"],
            alpha=cfg["alpha"],
            lr=cfg["lr"],
            sigma=cfg["sigma"],
            max_iter=cfg["max_iter"],
            tol=cfg["tol"],
            n_iter_no_change=cfg["ninc"]
        )
        m.fit(X, y)
        models.append(m)
    return models


def ensemble_predict(models, X):
    preds = np.vstack([m.predict(X) for m in models])
    return preds.mean(axis=0), preds.std(axis=0, ddof=1) + 1e-9


models = fit_ensemble(X, y, best_cfg)

# ============================================================
# 6. ACQUISITION (LOCAL MAXIMA PUSH)
# ============================================================

def erf_vec(x):
    return np.vectorize(math.erf)(x)

def compute_ei(mu, std, y_best, xi=0.001):
    z = (mu - y_best - xi) / std
    pdf = np.exp(-0.5*z*z) / np.sqrt(2*np.pi)
    cdf = 0.5 * (1 + erf_vec(z / np.sqrt(2)))
    return (mu - y_best - xi) * cdf + std * pdf


idx_best = np.argmax(y)
x_best, y_best = X[idx_best], y[idx_best]

rng = np.random.default_rng(2026)

Xcand = np.vstack([
    rng.uniform(0,1,(20000,2)),
    np.clip(rng.normal(x_best, 0.02, (20000,2)), 0, 1)
])

mu, std = ensemble_predict(models, Xcand)
ei = compute_ei(mu, std, y_best)
ucb = mu + 2.0 * std

score = 0.6 * (ucb - ucb.mean())/ucb.std() + 0.4 * (ei - ei.mean())/ei.std()

dists = np.sqrt(((Xcand[:,None,:]-X[None,:,:])**2).sum(axis=2))
min_dist = dists.min(axis=1)

valid = np.where(min_dist >= 0.012)[0]
best_idx = valid[np.argmax(score[valid])]

# ============================================================
# 7. FINAL 6-DECIMAL OUTPUT
# ============================================================

x_next = np.round(Xcand[best_idx], 6)

print("================================================")
print("WEEK 8 FUNCTION 2 — NEXT DATA POINT (6 DECIMALS)")
print("================================================")
print(f"x_best = {np.round(x_best,6)}, y_best = {y_best:.6f}")
print(f"x_next = {x_next}")
print(f"μ(x_next) = {mu[best_idx]:.6f}")
print(f"σ(x_next) = {std[best_idx]:.6f}")
print(f"EI        = {ei[best_idx]:.6f}")
print(f"min_dist  = {min_dist[best_idx]:.6f}")


WEEK 8 FUNCTION 2 — NEXT DATA POINT (6 DECIMALS)
x_best = [0.683406 0.063769], y_best = 0.629731
x_next = [0.007578 0.977359]
μ(x_next) = 0.180755
σ(x_next) = 0.376117
EI        = 0.021259
min_dist  = 0.520452
